<a href="https://colab.research.google.com/github/ajaymurali1998/aave-liquidation-pipeline/blob/main/aave_liquidation_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-cloud-bigquery eth-abi web3 pandas pyarrow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 72.2 MB/s eta 0:00:00


In [2]:
from google.cloud import bigquery
import os

In [4]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/content/ajay-blockchain-projects-cbdc74666d9d.json'
# Replace 'your-key-file.json' with actual filename
os.environ['GOOGLE_CLOUD_PROJECT'] = 'ajay-blockhain-projects'

client = bigquery.Client(project='ajay-blockhain-projects')
print('Connected to BigQuery')


Connected to BigQuery


In [5]:
import os
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/content/ajay-blockchain-projects-cbdc74666d9d.json',
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

client = bigquery.Client(
    project='ajay-blockchain-projects',
    credentials=credentials
)

print(f'Connected: {client.project}')

Connected: ajay-blockchain-projects



**Pull Raw Logs from BigQuery**

In [6]:
query = """
SELECT
  block_timestamp,
  block_number,
  transaction_hash,
  topics,
  data
FROM `bigquery-public-data.crypto_ethereum.logs`
WHERE lower(address) = lower('0x7d2768dE32b0b80b7a3454c06BdAc94A69DDc7A9')
AND topics[OFFSET(0)] = '0xe413a321e8681d831f4dbccbca790d2952b56f977908e45be37335533e005286'

  AND DATE(block_timestamp)>= '2025-01-01' AND DATE(block_timestamp)< '2025-07-01'
"""

df_raw = client.query(query).to_dataframe()
print(f'Fetched {len(df_raw)} raw liquidation events')


Fetched 7195 raw liquidation events


In [8]:
df_raw

,block_timestamp,block_number,transaction_hash,topics,data
0,2025-04-26 23:55:23+00:00,22356635,0xe6ed43eea3edb81d4ea38a6c1a7eccbadfeffec665cf...,[0xe413a321e8681d831f4dbccbca790d2952b56f97790...,0x00000000000000000000000000000000000000000000...
1,2025-05-03 03:50:23+00:00,22400684,0xfe27f1e57a0c71637df8099c4ac27d4054469ad4265d...,[0xe413a321e8681d831f4dbccbca790d2952b56f97790...,0x00000000000000000000000000000000000000000000...
2,2025-05-03 04:24:35+00:00,22400854,0x8b0cff0051ff8d649f2fe5ae3c368b504e5756dfc067...,[0xe413a321e8681d831f4dbccbca790d2952b56f97790...,0x00000000000000000000000000000000000000000000...
3,2025-05-03 07:08:47+00:00,22401663,0x7000c8d2e0f490d197cdf344d7a37f5b8fae7060f230...,[0xe413a321e8681d831f4dbccbca790d2952b56f97790...,0x00000000000000000000000000000000000000000000...
4,2025-05-03 07:08:47+00:00,22401663,0x7000c8d2e0f490d197cdf344d7a37f5b8fae7060f230...,[0xe413a321e8681d831f4dbccbca790d2952b56f97790...,0x00000000000000000000000000000000000000000000...
...,...,...,...,...,...
7190,2025-04-07 09:20:23+00:00,22216091,0x06485a3df9c5e266799336de9e9f82b889ddd4af115d...,[0xe413a321e8681d831f4dbccbca790d2952b56f97790...,0x00000000000000000000000000000000000000000000...
7191,2025-04-07 09:24:59+00:00,22216114,0xb52644875ecf2240498f3dfbdf52e4219c7e96e03a26...,[0xe413a321e8681d831f4dbccbca790d2952b56f97790...,0x00000000000000000000000000000000000000000000...
7192,2025-04-07 22:43:35+00:00,22220077,0xad28a0990015e6f9512a9b1395d24872cd6bb690fda5...,[0xe413a321e8681d831f4dbccbca790d2952b56f97790...,0x00000000000000000000000000000000000000000000...
7193,2025-04-07 23:53:59+00:00,22220425,0x2fb965963cb5d077346202f3f4b02c97ddf11a08e0fc...,[0xe413a321e8681d831f4dbccbca790d2952b56f97790...,0x00000000000000000000000000000000000000000000...



**ABI Decoder Function**

In [9]:
from eth_abi import decode
from web3 import Web3


In [10]:
def decode_liquidation_call(row):
    try:
        # Decode indexed topics (addresses are zero-padded to 32 bytes)
        collateral_asset = '0x' + row['topics'][1][-40:]
        debt_asset       = '0x' + row['topics'][2][-40:]
        borrower         = '0x' + row['topics'][3][-40:]

        # Decode non-indexed data field
        data_bytes = bytes.fromhex(row['data'][2:])
        decoded = decode(
            ['uint256', 'uint256', 'address', 'bool'],
            data_bytes
        )

        return {
            'block_timestamp':           row['block_timestamp'],
            'block_number':              row['block_number'],
            'transaction_hash':          row['transaction_hash'],
            'collateral_asset':          Web3.to_checksum_address(collateral_asset),
            'debt_asset':                Web3.to_checksum_address(debt_asset),
            'borrower':                  Web3.to_checksum_address(borrower),
            'debt_to_cover':             decoded[0],
            'liquidated_collateral_amt': decoded[1],
            'liquidator':                decoded[2],
            'receive_atoken':            decoded[3]
        }
    except Exception as e:
        return None


**Run Decoder + Write to BigQuery**

In [11]:
import pandas as pd

# Decode all rows
decoded_rows = [decode_liquidation_call(row) for _, row in df_raw.iterrows()]
df_decoded = pd.DataFrame([r for r in decoded_rows if r is not None])
print(f'Decoded {len(df_decoded)} rows successfully')




Decoded 7195 rows successfully


In [12]:
df_decoded.head()

,block_timestamp,block_number,transaction_hash,collateral_asset,debt_asset,borrower,debt_to_cover,liquidated_collateral_amt,liquidator,receive_atoken
0,2025-04-26 23:55:23+00:00,22356635,0xe6ed43eea3edb81d4ea38a6c1a7eccbadfeffec665cf...,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0x4Fabb145d64652a948d72533023f6E7A623C7C53,0xb33d01dD954888Ae2FdA24403e64b2e1daD84DFF,17721374258915282521,10214634691795186,0x64127e89204a5fea60ac6eaccf0eef0d4d7790a7,False
1,2025-05-03 03:50:23+00:00,22400684,0xfe27f1e57a0c71637df8099c4ac27d4054469ad4265d...,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xaaD566772812a743E7693a563827856536678C7e,9414050,5387488221865335,0x64127e89204a5fea60ac6eaccf0eef0d4d7790a7,False
2,2025-05-03 04:24:35+00:00,22400854,0x8b0cff0051ff8d649f2fe5ae3c368b504e5756dfc067...,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0x5c011516e9Cef971975bC39Da15c47b10206F23C,9385204,5370980184915465,0xd4bc53434c5e12cb41381a556c3c47e1a86e80e3,False
3,2025-05-03 07:08:47+00:00,22401663,0x7000c8d2e0f490d197cdf344d7a37f5b8fae7060f230...,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0x76E7A5bb155Ab6838caAdCf4AeC0a2620d7A97a3,3689497,3855524,0xd4bc53434c5e12cb41381a556c3c47e1a86e80e3,False
4,2025-05-03 07:08:47+00:00,22401663,0x7000c8d2e0f490d197cdf344d7a37f5b8fae7060f230...,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xdAC17F958D2ee523a2206206994597C13D831ec7,0x76E7A5bb155Ab6838caAdCf4AeC0a2620d7A97a3,5503909,5753277,0xd4bc53434c5e12cb41381a556c3c47e1a86e80e3,False


In [15]:
df_decoded['debt_to_cover'] = df_decoded['debt_to_cover'].astype(str)
df_decoded['liquidated_collateral_amt'] = df_decoded['liquidated_collateral_amt'].astype(str)

In [16]:
# Write to BigQuery
table_id = 'ajay-blockchain-projects.aave_decoded.liquidation_events'
job = client.load_table_from_dataframe(df_decoded, table_id)
job.result()
print(f'Written to {table_id}')

Written to ajay-blockchain-projects.aave_decoded.liquidation_events


In [19]:
df_decoded.head()

,block_timestamp,block_number,transaction_hash,collateral_asset,debt_asset,borrower,debt_to_cover,liquidated_collateral_amt,liquidator,receive_atoken
0,2025-04-26 23:55:23+00:00,22356635,0xe6ed43eea3edb81d4ea38a6c1a7eccbadfeffec665cf...,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0x4Fabb145d64652a948d72533023f6E7A623C7C53,0xb33d01dD954888Ae2FdA24403e64b2e1daD84DFF,17721374258915282521,10214634691795186,0x64127e89204a5fea60ac6eaccf0eef0d4d7790a7,False
1,2025-05-03 03:50:23+00:00,22400684,0xfe27f1e57a0c71637df8099c4ac27d4054469ad4265d...,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xaaD566772812a743E7693a563827856536678C7e,9414050,5387488221865335,0x64127e89204a5fea60ac6eaccf0eef0d4d7790a7,False
2,2025-05-03 04:24:35+00:00,22400854,0x8b0cff0051ff8d649f2fe5ae3c368b504e5756dfc067...,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0x5c011516e9Cef971975bC39Da15c47b10206F23C,9385204,5370980184915465,0xd4bc53434c5e12cb41381a556c3c47e1a86e80e3,False
3,2025-05-03 07:08:47+00:00,22401663,0x7000c8d2e0f490d197cdf344d7a37f5b8fae7060f230...,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0x76E7A5bb155Ab6838caAdCf4AeC0a2620d7A97a3,3689497,3855524,0xd4bc53434c5e12cb41381a556c3c47e1a86e80e3,False
4,2025-05-03 07:08:47+00:00,22401663,0x7000c8d2e0f490d197cdf344d7a37f5b8fae7060f230...,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xdAC17F958D2ee523a2206206994597C13D831ec7,0x76E7A5bb155Ab6838caAdCf4AeC0a2620d7A97a3,5503909,5753277,0xd4bc53434c5e12cb41381a556c3c47e1a86e80e3,False
